# Quickstart: parsing J&K electoral roll PDFs

This notebook shows how to call the parser directly as a Python function, without going through the command line. It assumes:

- you've already run `pip install -r requirements.txt` in this environment (ideally the Jupyter kernel is using the project's `.venv`), and
- the system dependencies (`poppler`, and `tesseract` if using the tesseract parser) are installed -- see the main [README.md](../README.md) for per-OS commands.

Two parser modules are available under `scripts/`:

| module | OCR engine | speed | accuracy |
|---|---|---|---|
| `parse_jk` | Tesseract | fast (~5-10s/page) | good, some blank/misread fields |
| `parse_jk_paddle` | PaddleOCR | slow (~30-60s/page, CPU) | noticeably more accurate |

Both expose the same function signature: `process_pdf(pdf_path, out_csv) -> pandas.DataFrame`.

In [ ]:
import sys
from pathlib import Path

# Make the sibling scripts/ folder importable from this notebook.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

import pandas as pd

## 1. Single file

Pick one parser module and call `process_pdf` directly. This can take anywhere from a couple of minutes (Tesseract) to well over half an hour (PaddleOCR) for a full ~40-page roll, so it's worth trying the faster Tesseract path first while iterating.

In [ ]:
from parse_jk import process_pdf  # Tesseract-based; swap for `from parse_jk_paddle import process_pdf` for PaddleOCR

pdf_path = PROJECT_ROOT / 'samples' / '2024-EROLLGEN-U08-79-FinalRoll-Revision2-ENG-8-WI.pdf'
out_csv = PROJECT_ROOT / 'output' / 'jk_79_8.csv'
out_csv.parent.mkdir(parents=True, exist_ok=True)

df = process_pdf(str(pdf_path), str(out_csv))
df.head()

## 2. Multiple files

Loop over every PDF in a folder and write one CSV per input, then optionally concatenate them into a single combined DataFrame.

In [ ]:
samples_dir = PROJECT_ROOT / 'samples'
out_dir = PROJECT_ROOT / 'output'
out_dir.mkdir(parents=True, exist_ok=True)

frames = []
pdfs = sorted(samples_dir.glob('*.pdf'))
for i, pdf in enumerate(pdfs, 1):
    out_path = out_dir / (pdf.stem + '.csv')
    print(f'[{i}/{len(pdfs)}] {pdf.name} -> {out_path}')
    frames.append(process_pdf(str(pdf), str(out_path)))

combined = pd.concat(frames, ignore_index=True)
combined.to_csv(out_dir / 'combined.csv', index=False)
print(f'{len(combined)} total elector records across {len(pdfs)} files')

## 3. Inspect the output

Basic sanity checks worth running on any new batch: row counts, and how often key fields come back blank.

In [ ]:
df = pd.read_csv(out_csv, dtype=str).fillna('')
print('rows:', len(df))

for col in ['number', 'id', 'elector_name', 'father_or_husband_name', 'house_no', 'age', 'sex']:
    blank = (df[col] == '').sum()
    print(f'{col:25s} blank: {blank}/{len(df)} ({100*blank/len(df):.1f}%)')

df.head(10)